# R4-N1 LR-A calibration (Protocol A)

This notebook reproduces LR-A calibration through the existing shared R4-N1 implementation. It fits on calibration-role recordings, persists the selected mapping and fit evidence, and only then opens operating predictions. It never reads the sealed test partition.

The completed canonical run is runs/r4/r4_n1_lr/. The default mode never executes into that canonical location: it reports the existing run or stops if the expected completed run is missing. To create a separate reproduction, explicitly set RUN_MODE = 'separate' and assign SEPARATE_RUN_DIR to a new, nonexisting directory. The notebook never deletes or overwrites existing evidence.

## 1. Mount Google Drive and locate the project

Expected project root: MyDrive/PAI&AIES Project. The canonical output_root in the recipe remains unchanged. A separate run requires an explicit, unique output path.

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
PROJECT_ROOT = Path('/content/drive/MyDrive/PAI&AIES Project').resolve()
CONFIG_PATH = PROJECT_ROOT / 'configs/r4_n1_calibration_a.json'
CANONICAL_RUN_DIR = PROJECT_ROOT / 'runs/r4/r4_n1_lr'

# Safe default: leave in canonical mode to refuse reuse of completed LR-A results.
RUN_MODE = 'canonical'  # Change only for an intentional separate reproduction.
SEPARATE_RUN_DIR = None  # Example: PROJECT_ROOT / 'runs/r4/r4_n1_lr_reproduction_20261115'

if RUN_MODE == 'canonical':
    RUN_DIR = CANONICAL_RUN_DIR
    if RUN_DIR.exists():
        raise FileExistsError(
            f'Completed canonical LR-A output already exists: {RUN_DIR}. '
            "Refusing to rerun. Set RUN_MODE='separate' and explicitly configure a new SEPARATE_RUN_DIR."
        )
    raise FileNotFoundError(
        f'Approved canonical LR-A results were expected at {RUN_DIR} but are absent. '
        "The notebook will not recreate the canonical run; use a separately configured output path."
    )
elif RUN_MODE == 'separate':
    if SEPARATE_RUN_DIR is None:
        raise ValueError('Set SEPARATE_RUN_DIR to an explicit new output path before using separate mode.')
    RUN_DIR = Path(SEPARATE_RUN_DIR)
    RUN_DIR = RUN_DIR if RUN_DIR.is_absolute() else PROJECT_ROOT / RUN_DIR
    RUN_DIR = RUN_DIR.resolve()
    if RUN_DIR == CANONICAL_RUN_DIR.resolve():
        raise ValueError('Separate mode must use a path different from the canonical LR-A run.')
    if RUN_DIR.exists():
        raise FileExistsError(f'Refusing to overwrite existing reproduction directory: {RUN_DIR}')
else:
    raise ValueError("RUN_MODE must be 'canonical' or 'separate'.")

required_project_files = [
    PROJECT_ROOT / 'src/echoproof/evaluation/r4_n1.py',
    PROJECT_ROOT / 'scripts/r4_n1_calibrate.py',
    CONFIG_PATH,
]
missing = [str(path) for path in required_project_files if not path.is_file()]
if missing:
    raise FileNotFoundError('Missing project files in Drive:\n' + '\n'.join(missing))
print(f'Project root: {PROJECT_ROOT}')
print(f'Frozen recipe: {CONFIG_PATH}')
print(f'Canonical approved results: {CANONICAL_RUN_DIR}')
print(f'New run destination: {RUN_DIR}')
print('Destination is absent; no existing run will be overwritten.')

## 2. Check dependencies and import the project implementation

The notebook imports EchoProof from the project's src tree and uses its calibration runner. It installs only missing calibration dependencies; it does not upgrade installed packages or load the LR model for training.

In [ ]:
import importlib.util
import subprocess
import sys

if sys.version_info < (3, 11):
    raise RuntimeError(f'EchoProof requires Python 3.11 or newer; Colab has {sys.version.split()[0]}')
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
required_modules = {
    'numpy': 'numpy', 'scipy': 'scipy', 'pyarrow': 'pyarrow',
    'matplotlib': 'matplotlib', 'sklearn': 'scikit-learn',
}
missing_packages = [package for module, package in required_modules.items() if importlib.util.find_spec(module) is None]
if missing_packages:
    print('Installing missing packages only:', missing_packages)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check', *missing_packages])
else:
    print('All calibration dependencies are already installed.')

from echoproof.evaluation import r4_n1
from echoproof.artifacts import file_digest
print('Imported R4-N1 module:', r4_n1.__file__)
print('Python:', sys.version.replace('\n', ' '))

## 3. Audit LR-A selection and calibration inputs

Inputs come from runs/r4/r4_m1_a_v1_lr/ and runs/r4/r4_m1_a_v1_lr_export/. The audit validates model/evidence hashes, selection and calibration recording/chunk Parquet hashes and schemas, role counts, labels, component counts, recording/chunk score parity, and speaker separation. It checks operating-file presence only; operating prediction contents and hashes are deferred until after the mapping is frozen. It does not access test data.

In [ ]:
recipe = r4_n1.load_recipe(CONFIG_PATH)
if recipe['models']['lr']['verification_state'] != 'verified_local':
    raise RuntimeError('Recipe does not mark LR-A inputs as verified.')
audit = r4_n1.audit_inputs(PROJECT_ROOT, recipe, model_name='lr')
print('Selection/calibration preflight checks:')
for check, passed in audit.get('checks', {}).items():
    print(f"  {'PASS' if passed else 'FAIL'}  {check}")
if not audit.get('passed'):
    raise RuntimeError('LR-A input audit failed:\n' + '\n'.join(audit.get('problems', [])))
print('Validated role summaries:', audit['roles'])
print('Chunk-to-recording mean-score maximum differences:', audit['chunk_mean_max_abs_difference'])
print('Operating prediction contents read:', audit['operating_predictions_content_read'])
print('R4-S1 manifest verification:', audit['r4_s1_manifest_verification'])
print('Preflight passed; no mapping has been fitted yet.')

## 4. Execute LR-A calibration

This is the only calibration cell. It calls scripts/r4_n1_calibrate.py with --model lr and the frozen recipe. The runner fits identity, temperature, and affine mappings on calibration recordings only; performs leave-one-speaker-out selection; persists and hash-checks the frozen fit stage; and only then opens and validates operating predictions for acceptance and threshold selection. The recipe and calibration rules are not changed.

In [ ]:
import importlib.metadata
import json
from datetime import datetime, timezone

if RUN_DIR.exists():
    raise FileExistsError(f'Refusing to overwrite existing LR-A output: {RUN_DIR}')
started_utc = datetime.now(timezone.utc).isoformat()
command = [
    sys.executable, str(PROJECT_ROOT / 'scripts/r4_n1_calibrate.py'),
    '--model', 'lr', '--root', str(PROJECT_ROOT), '--config', str(CONFIG_PATH),
    '--run-dir', str(RUN_DIR),
]
print('Starting LR-A calibration into:', RUN_DIR)
completed = subprocess.run(command, cwd=PROJECT_ROOT, text=True, capture_output=True)
print(completed.stdout)
if completed.stderr:
    print(completed.stderr, file=sys.stderr)
if completed.returncode != 0:
    partial_run = str(RUN_DIR) if RUN_DIR.exists() else 'no run directory created'
    raise RuntimeError(f"LR-A calibration failed with exit code {completed.returncode}; inspect {partial_run}. Existing results are not removed or overwritten.")
if not (RUN_DIR / 'fit/stage_complete.json').is_file() or not (RUN_DIR / 'operating/stage_complete.json').is_file():
    raise RuntimeError('Runner returned success without both stage completion records.')
completed_utc = datetime.now(timezone.utc).isoformat()
execution = {
    'model': 'lr', 'protocol': 'A', 'python': sys.version, 'platform': sys.platform,
    'started_utc': started_utc, 'completed_utc': completed_utc,
    'packages': {name: importlib.metadata.version(name) for name in ['numpy', 'scipy', 'pyarrow', 'matplotlib', 'scikit-learn']},
    'command': command, 'project_root': str(PROJECT_ROOT), 'recipe_sha256': file_digest(CONFIG_PATH),
    'output_directory': str(RUN_DIR), 'runner_returncode': completed.returncode, 'test_data_read': False,
}
execution_path = RUN_DIR / 'execution_environment.json'
with execution_path.open('x', encoding='utf-8') as stream:
    json.dump(execution, stream, indent=2, sort_keys=True)
    stream.write('\n')
print('LR-A calibration completed. Execution record:', execution_path)

## 5. Review calibration, operating metrics, diagnostics, and evidence

This cell reads only the selected run's generated R4-N1 reports. It summarizes the selected mapping, LOSO results and exclusions, operating acceptance, score type, threshold and plateau, probability metrics, diagnostics, bootstrap intervals, reliability plot, and generated evidence files.

In [ ]:
fit_report = json.loads((RUN_DIR / 'fit/calibration.json').read_text(encoding='utf-8'))
loso_report = json.loads((RUN_DIR / 'fit/loso_comparison.json').read_text(encoding='utf-8'))
operating_report = json.loads((RUN_DIR / 'operating/operating_results.json').read_text(encoding='utf-8'))
print('Run output directory:', RUN_DIR)
print('Selected mapping and parameters:', fit_report['selected_mapping'], fit_report['selected_parameters'])
print('LOSO selection summary:', fit_report['selection'])
print('LOSO candidate failures:', loso_report['failures'])
print('Calibration accepted:', operating_report['mapping_accepted'])
print('Deployed score type:', operating_report['score_type'])
print('Threshold and decision:', operating_report['threshold_rule'])
print('Threshold metrics and tied plateau:', operating_report['threshold_metrics'])
for view, metrics in operating_report['probability_metrics'].items():
    print(f"{view}: AUC={metrics['roc_auc']:.6f}; EER={metrics['eer']:.6f}; Macro-F1@0.5={metrics['macro_f1']:.6f}; log-loss={metrics['log_loss']:.6f}; Brier={metrics['brier']:.6f}; ECE={metrics['ece']:.6f}")
print('Selected-threshold macro F1:', operating_report['threshold_metrics']['macro_f1'])
print('Per-generator diagnostics:', sorted(operating_report['per_component']))
print('Per-speaker false-alarm rows:', len(operating_report['per_speaker_false_alarms']))
print('Speaker bootstrap:', operating_report['speaker_bootstrap'])
print('Reliability plot:', RUN_DIR / operating_report['reliability_plot']['path'])
print('Generated evidence:')
for path in sorted(RUN_DIR.rglob('*')):
    if path.is_file():
        print(f'  {path.relative_to(RUN_DIR)} ({path.stat().st_size} bytes)')